# Lekcia 12 - Skrátenie histórie chatu pomocou agenta Scratchpad

Táto poznámková kniha demonštruje, ako spravovať kontext v dlhých rozhovoroch pomocou Microsoft Agent Framework. Ako sa rozhovory rozrastajú, počet tokenov rastie — nakoniec prekročí kontextové okno modelu. Riešime to pomocou **vzoru zhrnutia kontextu** a **agenta scratchpad** pre trvalú pamäť.

## Čo sa naučíte:
1. **Prečo je správa kontextu dôležitá**: Pochopenie limitov tokenov a kontextových okien
2. **Agent s vedomím kontextu**: Vytváranie agentov, ktorí spravujú vlastný kontext rozhovoru
3. **Vzor zhrnutia kontextu**: Použitie nástrojov na zhrnutie histórie rozhovoru
4. **Agent Scratchpad**: Trvalá pamäť, ktorá prežije skrátenie kontextu

## Predpoklady:
- Nastavenie Azure OpenAI s nakonfigurovanými environmentálnymi premennými
- Pochopenie základných konceptov agentov z predchádzajúcich lekcií


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Prečo je správa kontextu dôležitá

Každý LLM má obmedzené **kontextové okno** — maximálny počet tokenov, ktoré môže spracovať v jednom požiadavku. Ako sa vyvíja viackolová konverzácia:

- **Počet tokenov rastie lineárne** s každou správou používateľa a odpoveďou asistenta.
- **Tokeny výzvy dominujú nákladom**, pretože celá história sa odosiela znova pri každom kole.
- Nakoniec konverzácia **prekročí kontextové okno** a model buď obmedzí text, alebo vyhodí chybu.

### Stratégiie správy kontextu

| Stratégia | Ako funguje | Kompromis |
|---|---|---|
| **Obmedzenie (Truncation)** | Odstráni najstaršie správy | Stráca sa skorší kontext |
| **Zhrnutie (Summarization)** | Zhrnie staršie správy do súhrnu | Niektoré detaily sa stratia, ale kľúčové body zostanú |
| **Scratchpad / Externá pamäť** | Ukladá kľúčové fakty mimo konverzácie | Vyžaduje volania nástrojov, ale prežije akékoľvek zmenšenie |

V tomto notebooku kombinujeme **zhrnutie** s **nástrojom scratchpad**, aby agent mohol udržiavať kontinuitu aj pri kondenzovanej histórii konverzácie.


## Vytváranie agenta s kontextovým povedomím


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulácia dlhého rozhovoru

Poďme si prejsť viackolový rozhovor, aby sme videli, ako sa kontext hromadí. Agent by si mal zachovať kľúčové detaily (preferencie, rozpočet, dátumy cesty) počas jednotlivých kôl a preukázať kontinuitu.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Všimnite si, ako agent uchováva kontext z predchádzajúcich kôl — vie o Japonsku, suši, chrámoch, fotografovaní, rozpočte 3000 dolárov, sólo cestovaní a aprílovom termíne. V krátkom rozhovore to funguje dobre, ale ako sa rozhovor rozrastá, posielanie celej histórie sa stáva nákladným.

Pokračujme v rozhovore s viacerými kolami, aby sme videli kumuláciu kontextu:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzor zhrnutia kontextu

Ako sa rozhovor rozrastá, môžeme použiť **nástroj na zhrnutie**, ktorý skondenzuje nazbieraný kontext do kompaktného formátu. Agent volá tento nástroj na zaznamenanie kľúčových preferencií, takže aj keď staršie správy zmiznú, dôležité informácie sú zachované.

Tento vzor je stavebným kameňom pre sofistikovanejšie redukovanie histórie:
1. Agent identifikuje kľúčové fakty z rozhovoru
2. Zavolá nástroj na zhrnutie, aby ich uložil
3. Staršie správy možno bezpečne odstrániť, pretože súhrn zachytáva podstatné

Nižšie definujeme nástroj `summarize_preferences`, ktorý môže agent využiť na zaznamenanie kompaktného shrnutí toho, čo sa naučil.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Zhrnutie

V tejto lekcii ste sa naučili, ako spravovať kontext v dlhodobých konverzáciách agentov pomocou Microsoft Agent Frameworku:

### Kľúčové koncepty
- **Kontextové okná sú obmedzené** — každý token v histórii konverzácie stojí peniaze a započítava sa do limitu.
- **Nástroje na sumarizáciu** umožňujú agentovi skondenzovať nahromadený kontext do kompaktných zhrnutí, čím sa znižuje používanie tokenov pri zachovaní podstatných informácií.
- **Scratchpady agentov** poskytujú trvalú externú pamäť, ktorá pretrváva aj pri zmenšovaní konverzácie.

### Čo ste vybudovali
- **Agent so znalosťou kontextu**, ktorý udržiava kontinuitu v rámci viackolových konverzácií
- **Nástroj na sumarizáciu** (`summarize_preferences`), ktorý zaznamenáva kľúčové údaje o používateľovi v kompaktnom formáte
- **Viackolová konverzácia** demonštrujúca uchovávanie kontextu a spracovanie zmien

### Aplikácie v reálnom svete
- **Boti zákazníckej podpory**: Pamätajú si preferencie počas dlhých podporovacích relácií
- **Osobní asistenti**: Sledujú prebiehajúce projekty bez nutnosti opakovane vysvetľovať kontext
- **Vzdelávací tutoriály**: Udržiavajú pokrok študentov počas mnohých interakcií

### Ďalšie kroky
- Implementovať plnohodnotný nástroj scratchpadu s perzistenciou na základe súborov
- Pridať automatické skracovanie histórie po sumarizácii
- Kombinovať so vektorovými databázami pre sémantické vyhľadávanie v pamäti
- Vytvárať agentov, ktorí môžu po dňoch pokračovať v konverzáciách s úplným kontextom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
